# Getting Weather

## Loading Packages

In [1]:
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

import requests

In [2]:
output_dir = "../data"

In [3]:
citibike_df = pd.read_csv(f"{output_dir}/JC/JC2025.csv", parse_dates=['started_at', 'ended_at'])

In [4]:
lat = 40.7178
lng = -74.0431

start_date = '2025-01-01'
end_date = '2025-12-31'

url = 'https://archive-api.open-meteo.com/v1/archive'

params = {
    'latitude': lat,
    'longitude': lng,
    "start_date": start_date,
    "end_date": end_date,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "America/New_York"
    
}

response = requests.get(url, params=params)
response.raise_for_status()

In [5]:
data = response.json()

In [6]:
data['daily'].keys()

dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'wind_speed_10m_max'])

In [7]:
weather_data = pd.DataFrame(data['daily'])
weather_data.head()

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-01,11.0,3.9,7.4,4.5,4.5,0.0,23.2
1,2025-01-02,5.4,0.3,2.6,0.0,0.0,0.0,25.1
2,2025-01-03,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
3,2025-01-04,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1
4,2025-01-05,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9


In [8]:
weather_data.rename(columns={'time':'date'},inplace=True)
weather_data.to_csv(f'{output_dir}/JC/jersey_weather_2025.csv', index=False)

In [9]:
weather_daily = pd.read_csv(f'{output_dir}/JC/jersey_weather_2025.csv', parse_dates=['date'])
weather_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 365 non-null    datetime64[us]
 1   temperature_2m_max   365 non-null    float64       
 2   temperature_2m_min   365 non-null    float64       
 3   temperature_2m_mean  365 non-null    float64       
 4   precipitation_sum    365 non-null    float64       
 5   rain_sum             365 non-null    float64       
 6   snowfall_sum         365 non-null    float64       
 7   wind_speed_10m_max   365 non-null    float64       
dtypes: datetime64[us](1), float64(7)
memory usage: 22.9 KB


In [10]:
weather_daily.head()

,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-01,11.0,3.9,7.4,4.5,4.5,0.0,23.2
1,2025-01-02,5.4,0.3,2.6,0.0,0.0,0.0,25.1
2,2025-01-03,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
3,2025-01-04,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1
4,2025-01-05,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9


## Vizualization of Weather

### Daily Average Temperature Over Time

In [11]:
fig = px.line(
    weather_daily,
    x='date',
    y='temperature_2m_min',
    title='Daily Average Temperature Over Time',
    markers=False
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Average Temperature',
    hovermode='x unified'
)

fig.show()

In [12]:
weather_daily.head()

,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-01,11.0,3.9,7.4,4.5,4.5,0.0,23.2
1,2025-01-02,5.4,0.3,2.6,0.0,0.0,0.0,25.1
2,2025-01-03,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
3,2025-01-04,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1
4,2025-01-05,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9


In [13]:
temperature_long = weather_daily.melt(
    id_vars='date',
    value_vars=[
        'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean'
    ],
    
    var_name='temperature_type',
    value_name='temperature'
)

temperature_long.head()

,date,temperature_type,temperature
0,2025-01-01,temperature_2m_max,11.0
1,2025-01-02,temperature_2m_max,5.4
2,2025-01-03,temperature_2m_max,3.2
3,2025-01-04,temperature_2m_max,-0.1
4,2025-01-05,temperature_2m_max,0.4


In [14]:
temperature_long['temperature_type'] = temperature_long['temperature_type'].str.replace('temperature_', '')


In [15]:
temperature_long.head()

,date,temperature_type,temperature
0,2025-01-01,2m_max,11.0
1,2025-01-02,2m_max,5.4
2,2025-01-03,2m_max,3.2
3,2025-01-04,2m_max,-0.1
4,2025-01-05,2m_max,0.4


### Daily Average Temperature Over Time: Mean, Max, Min

In [16]:
fig = px.line(
    temperature_long,
    x='date',
    y='temperature',
    color='temperature_type',
    title='Daily Average Temperature Over Time: Mean, Max, Min',
    markers=False
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Average Temperature',
    hovermode='x unified'
)

fig.show()

In [17]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
